# 🧠⚡ Kernel Vision: Inside Neural Network Optimization

**Dissect a model → Watch it learn → See the compiler optimize it**

A visual journey through GPT-2 fine-tuning and `torch.compile` kernel fusion.

---

In [ ]:
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.colors as mcolors
import networkx as nx
import numpy as np
from scipy.ndimage import gaussian_filter
from IPython.display import clear_output, display, HTML
import time
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'💻 Device: {device} | PyTorch: {torch.__version__}')
if device == 'cuda':
    print(f'🎮 GPU: {torch.cuda.get_device_name()}')

plt.style.use('dark_background')
plt.rcParams.update({'figure.facecolor': '#0d1117', 'axes.facecolor': '#0d1117',
                     'savefig.facecolor': '#0d1117', 'font.family': 'monospace'})

display(HTML("""
<style>
  .output_area { background: #0d1117 !important; }
  .output_subarea { max-width: 100% !important; }
</style>
"""))

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained('gpt2', output_attentions=True).to(device)

num_layers = model.config.n_layer
num_heads = model.config.n_head
hidden_size = model.config.n_embd
head_dim = hidden_size // num_heads
num_params = sum(p.numel() for p in model.parameters()) / 1e6

PROBE = 'The transformer learned to fuse attention kernels for faster inference on the GPU'
probe_inputs = tokenizer(PROBE, return_tensors='pt').to(device)
probe_tokens = tokenizer.convert_ids_to_tokens(probe_inputs['input_ids'][0])

print(f'✅ GPT-2 loaded: {num_layers}L × {num_heads}H × {head_dim}d = {num_params:.0f}M params')
print(f'🔍 Probe: "{PROBE}"')

## The Weight Landscape — 3D Projection Kernel Surfaces
GPT-2 learns **query/key/value projection matrices** that transform tokens before attention.
Each surface below is a 64×64 slice of a projection weight matrix — the "terrain" the model navigates during inference.

In [ ]:
fig = plt.figure(figsize=(20, 5))
fig.patch.set_facecolor('#0d1117')

show_layers = [0, 3, 7, 11]
cmaps = ['inferno', 'plasma', 'magma', 'cividis']

for idx, (li, cmap) in enumerate(zip(show_layers, cmaps)):
    qkv = model.transformer.h[li].attn.c_attn.weight.detach().cpu().numpy()
    q_weights = qkv[:, :hidden_size]
    surface = q_weights[:64, :64]
    surface = gaussian_filter(surface, sigma=1.2)
    X, Y = np.meshgrid(range(64), range(64))

    ax = fig.add_subplot(1, 4, idx + 1, projection='3d')
    ax.plot_surface(X, Y, surface, cmap=cmap, alpha=0.9, antialiased=True,
                    rstride=2, cstride=2, edgecolor='none')
    ax.set_title(f'Layer {li} \u2014 Q Projection', fontsize=11, pad=10, color='white')
    ax.set_zticks([])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('#222')
    ax.yaxis.pane.set_edgecolor('#222')
    ax.zaxis.pane.set_edgecolor('#222')
    ax.view_init(elev=35, azim=45 + idx * 20)

fig.suptitle('\U0001f3d4\ufe0f  Q-Projection Weight Landscapes Across Depth',
             fontsize=16, fontweight='bold', color='#e6e6e6', y=1.02)
plt.tight_layout()
plt.show()
print('\nEach surface = 64\u00d764 slice of the Query projection matrix.')
print('Deeper layers show more complex terrain \u2014 richer learned representations.')

## Attention Optics — How the Network "Sees"
Each attention head focuses on different relationships between tokens.
Bright cells = strong attention. Diagonal = self/local. Off-diagonal = long-range dependencies.

In [ ]:
with torch.no_grad():
    outputs = model(**probe_inputs, output_attentions=True)

show_layers = [0, 3, 7, 11]
show_heads = [0, 3, 6, 10]
n_tokens = len(probe_tokens)

fig, axes = plt.subplots(len(show_layers), len(show_heads), figsize=(18, 15))
fig.patch.set_facecolor('#0d1117')

for row, layer_idx in enumerate(show_layers):
    attn = outputs.attentions[layer_idx][0]
    for col, head_idx in enumerate(show_heads):
        ax = axes[row, col]
        head_attn = attn[head_idx].cpu().numpy()
        im = ax.imshow(head_attn, cmap='magma', aspect='auto', vmin=0, vmax=0.4)
        if row == 0:
            ax.set_title(f'Head {head_idx}', fontsize=11, color='#ccc', pad=8)
        if col == 0:
            ax.set_ylabel(f'Layer {layer_idx}', fontsize=11, color='#ccc')
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_color('#333')

fig.suptitle('\U0001f52d Attention Optics \u2014 16 Views Into the Network\'s Mind',
             fontsize=16, fontweight='bold', color='#e6e6e6', y=1.01)
fig.text(0.5, -0.01, f'Probe: "{PROBE}"',
         ha='center', fontsize=10, color='#666', style='italic')
plt.tight_layout()
plt.show()

## Live Surgery — Watch the Network Rewire During Fine-Tuning
Four panels update in real-time as we fine-tune GPT-2:
- **Weight Δ** — How much each layer's weights have shifted
- **Gradient Flow** — Where learning is active (normalized gradient magnitude)
- **Loss Curve** — Training loss with exponential moving average
- **Attention Drift** — How attention patterns reorganize from the original

In [ ]:
# Snapshot initial state
with torch.no_grad():
    pre_out = model(**probe_inputs, output_attentions=True)
    init_attns = [a[0].cpu().clone() for a in pre_out.attentions]

init_weights = {}
for name, p in model.named_parameters():
    if 'attn' in name and 'weight' in name:
        init_weights[name] = p.detach().cpu().clone()

TRAIN_TEXT = """Yo, I'll tell you what I want, what I really, really want
So tell me what you want, what you really, really want
I'll tell you what I want, what I really, really want
So tell me what you want, what you really, really want
I wanna (hey!), I wanna (hey!), I wanna (hey!), I wanna (hey!)
I wanna really, really, really wanna "zig-a-zig", ah
If you want my future, forget my past
If you wanna get with me, better make it fast
Now don't go wasting my precious time
Get your act together, we could be just fine
I'll tell you what I want, what I really, really want
So tell me what you want, what you really, really want
I wanna (hey!), I wanna (hey!), I wanna (hey!), I wanna (hey!)
I wanna really, really, really wanna "zig-a-zig", ah
If you wanna be my lover, you gotta get with my friends (gotta get with my friends)
Make it last forever, friendship never ends
If you wanna be my lover, you have got to give
Taking is too easy, but that's the way it is"""

train_ids = tokenizer(TRAIN_TEXT, return_tensors='pt', truncation=True, max_length=512)['input_ids'].to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
NUM_STEPS = 60
VIZ_EVERY = 6

losses = []
ema_losses = []
attn_drifts = []
weight_deltas_history = []
grad_mags_history = []
ema_alpha = 0.15

fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('#0d1117')

print('\U0001f52c Live Fine-Tuning \u2014 Watching the network rewire...\n')

for step in range(NUM_STEPS):
    model.train()
    optimizer.zero_grad()
    out = model(train_ids, labels=train_ids)
    loss = out.loss
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    ema = losses[0] if step == 0 else ema_alpha * losses[-1] + (1 - ema_alpha) * ema_losses[-1]
    ema_losses.append(ema)

    # Collect gradient magnitudes per layer (normalized)
    grad_mags = []
    for name, p in model.named_parameters():
        if 'attn' in name and 'weight' in name and p.grad is not None:
            grad_mags.append((p.grad.norm() / (p.norm() + 1e-8)).item())
    grad_mags_history.append(grad_mags)

    if step % VIZ_EVERY == 0 or step == NUM_STEPS - 1:
        model.eval()
        with torch.no_grad():
            cur_out = model(**probe_inputs, output_attentions=True)
            cur_attns = [a[0].cpu() for a in cur_out.attentions]

        # Attention drift
        drift = sum((cur_attns[l] - init_attns[l]).abs().mean().item()
                     for l in range(num_layers)) / num_layers
        attn_drifts.append(drift)

        # Weight deltas
        deltas = []
        for name, p in model.named_parameters():
            if name in init_weights:
                deltas.append((p.detach().cpu() - init_weights[name]).abs().mean().item())
        weight_deltas_history.append(deltas)

        clear_output(wait=True)
        fig.clf()
        gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)

        # Panel 1: Weight delta heatmap
        ax1 = fig.add_subplot(gs[0, 0])
        wd = np.array(weight_deltas_history).T
        if wd.shape[1] > 1:
            ax1.imshow(wd, cmap='hot', aspect='auto', interpolation='bilinear')
        ax1.set_title('Weight \u0394 by Layer Over Time', fontsize=12, fontweight='bold', color='#ff6b6b')
        ax1.set_ylabel('Layer', fontsize=10, color='#aaa')
        ax1.set_xlabel('Checkpoint', fontsize=10, color='#aaa')
        ax1.tick_params(colors='#666')

        # Panel 2: Gradient flow
        ax2 = fig.add_subplot(gs[0, 1])
        if grad_mags:
            colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(grad_mags)))
            ax2.barh(range(len(grad_mags)), grad_mags, color=colors, edgecolor='none')
        ax2.set_title('Gradient Flow (||grad||/||weight||)', fontsize=12, fontweight='bold', color='#bd93f9')
        ax2.set_xlabel('Relative Magnitude', fontsize=10, color='#aaa')
        ax2.set_ylabel('Attn Layer', fontsize=10, color='#aaa')
        ax2.tick_params(colors='#666')
        ax2.invert_yaxis()

        # Panel 3: Loss curve
        ax3 = fig.add_subplot(gs[1, 0])
        ax3.plot(losses, color='#00d4ff', linewidth=1, alpha=0.4, label='Raw')
        ax3.plot(ema_losses, color='#00d4ff', linewidth=2.5, label='EMA')
        ax3.fill_between(range(len(ema_losses)), ema_losses, alpha=0.1, color='#00d4ff')
        ax3.set_title('Training Loss', fontsize=12, fontweight='bold', color='#00d4ff')
        ax3.set_xlabel('Step', fontsize=10, color='#aaa')
        ax3.set_ylabel('Loss', fontsize=10, color='#aaa')
        ax3.legend(fontsize=9, loc='upper right')
        ax3.grid(alpha=0.1)
        ax3.tick_params(colors='#666')

        # Panel 4: Attention drift
        ax4 = fig.add_subplot(gs[1, 1])
        drift_x = list(range(0, len(attn_drifts) * VIZ_EVERY, VIZ_EVERY))[:len(attn_drifts)]
        ax4.fill_between(drift_x, attn_drifts, alpha=0.2, color='#e63946')
        ax4.plot(drift_x, attn_drifts, color='#e63946', linewidth=2.5, marker='o', markersize=4)
        ax4.set_title('Attention Drift from Original', fontsize=12, fontweight='bold', color='#e63946')
        ax4.set_xlabel('Step', fontsize=10, color='#aaa')
        ax4.set_ylabel('\u0394 Attention', fontsize=10, color='#aaa')
        ax4.grid(alpha=0.1)
        ax4.tick_params(colors='#666')

        fig.suptitle(f'Step {step+1}/{NUM_STEPS}  \u2502  Loss: {losses[-1]:.4f}  \u2502  '
                     f'Attention \u0394: {drift:.5f}',
                     fontsize=13, color='#888', y=1.01)
        display(fig)

print(f'\n\u2705 Fine-tuning complete!')
print(f'   Loss: {losses[0]:.3f} \u2192 {losses[-1]:.3f}')
print(f'   Attention drift: {attn_drifts[0]:.5f} \u2192 {attn_drifts[-1]:.5f} ({attn_drifts[-1]/max(attn_drifts[0],1e-8):.1f}x)')

## The Showdown — Compiled vs Eager, Head-to-Head
We compile the fine-tuned model with `torch.compile` and race it against vanilla eager mode.
`torch.compile` uses **Triton** to fuse GPU kernels — fewer kernel launches, less memory traffic, more speed.

In [ ]:
model.eval()
gpu_name = torch.cuda.get_device_name() if device == 'cuda' else 'N/A'
print(f'\u23f3 Compiling model with torch.compile (max-autotune)...')
compiled_model = torch.compile(model, mode='max-autotune')

# Warm up compiled model (triggers Triton compilation + CUDA graph capture)
warmup_input = tokenizer(PROBE, return_tensors='pt').to(device)
for _ in range(15):
    with torch.no_grad():
        _ = compiled_model(**warmup_input)
for _ in range(15):
    with torch.no_grad():
        _ = model(**warmup_input)
if device == 'cuda': torch.cuda.synchronize()
print(f'\u2705 Compiled on {gpu_name}! Racing...\n')

In [ ]:
bench_input = tokenizer(PROBE, return_tensors='pt').to(device)
NUM_BATCHES = 40

print('Benchmarking compiled...')
compiled_times = []
for _ in range(NUM_BATCHES):
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = compiled_model(**bench_input)
    if device == 'cuda': torch.cuda.synchronize()
    compiled_times.append(time.perf_counter() - t0)

print('Benchmarking eager...')
eager_times = []
for _ in range(NUM_BATCHES):
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model(**bench_input)
    if device == 'cuda': torch.cuda.synchronize()
    eager_times.append(time.perf_counter() - t0)

eager_cum = np.cumsum(eager_times)
compiled_cum = np.cumsum(compiled_times)
max_time = max(eager_cum[-1], compiled_cum[-1])

# \u2500\u2500 ANIMATE \u2500\u2500
fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('#0d1117')
FRAMES = 50

for frame in range(FRAMES + 1):
    t = (frame / FRAMES) * max_time
    eager_done = min(int(np.searchsorted(eager_cum, t, side='right')), NUM_BATCHES)
    compiled_done = min(int(np.searchsorted(compiled_cum, t, side='right')), NUM_BATCHES)

    clear_output(wait=True)
    fig.clf()
    gs = fig.add_gridspec(4, 1, height_ratios=[1.2, 1.2, 0.3, 1.8], hspace=0.5)

    # Compiled bar
    ax_c = fig.add_subplot(gs[0])
    ax_c.barh([0], [NUM_BATCHES], height=0.6, color='#0d1117', edgecolor='#333', linewidth=1)
    bar_color = '#51cf66' if compiled_done < NUM_BATCHES else '#ffd700'
    ax_c.barh([0], [compiled_done], height=0.6, color=bar_color, edgecolor='white', linewidth=2)
    ax_c.set_xlim(0, NUM_BATCHES * 1.3)
    ax_c.set_yticks([])
    ax_c.set_title('\U0001f680  torch.compile + Triton Fused Kernels', fontsize=16,
                    fontweight='bold', color='#51cf66', loc='left')
    ax_c.text(compiled_done + 0.5, 0, f'  {compiled_done}/{NUM_BATCHES}',
             va='center', fontsize=18, fontweight='bold', color='#51cf66')
    if compiled_done > 0 and t > 0:
        ax_c.text(NUM_BATCHES * 1.1, 0, f'{compiled_done/t:.0f} batch/s',
                  va='center', fontsize=14, color='#666')
    ax_c.spines[['top', 'right', 'bottom']].set_visible(False)
    ax_c.tick_params(bottom=False, labelbottom=False)

    # Eager bar
    ax_e = fig.add_subplot(gs[1])
    ax_e.barh([0], [NUM_BATCHES], height=0.6, color='#0d1117', edgecolor='#333', linewidth=1)
    ax_e.barh([0], [eager_done], height=0.6, color='#ff6b6b', edgecolor='white', linewidth=2)
    ax_e.set_xlim(0, NUM_BATCHES * 1.3)
    ax_e.set_yticks([])
    ax_e.set_title('\U0001f40c  Eager Mode (vanilla PyTorch)', fontsize=16,
                    fontweight='bold', color='#ff6b6b', loc='left')
    ax_e.text(eager_done + 0.5, 0, f'  {eager_done}/{NUM_BATCHES}',
             va='center', fontsize=18, fontweight='bold', color='#ff6b6b')
    if eager_done > 0 and t > 0:
        ax_e.text(NUM_BATCHES * 1.1, 0, f'{eager_done/t:.0f} batch/s',
                  va='center', fontsize=14, color='#666')
    ax_e.spines[['top', 'right', 'bottom']].set_visible(False)
    ax_e.tick_params(bottom=False, labelbottom=False)

    # Timer
    ax_t = fig.add_subplot(gs[2])
    ax_t.axis('off')
    ax_t.text(0.5, 0.5, f'\u23f1  {t:.3f}s elapsed',
             ha='center', va='center', fontsize=16, color='#555',
             transform=ax_t.transAxes)

    # Throughput chart
    ax_ch = fig.add_subplot(gs[3])
    t_e = np.concatenate([[0], eager_cum[:eager_done]])
    t_c = np.concatenate([[0], compiled_cum[:compiled_done]])
    b_e = np.arange(eager_done + 1)
    b_c = np.arange(compiled_done + 1)
    ax_ch.fill_between(t_c, b_c, alpha=0.12, color='#51cf66')
    ax_ch.fill_between(t_e, b_e, alpha=0.12, color='#ff6b6b')
    ax_ch.plot(t_c, b_c, color='#51cf66', linewidth=3, label='Compiled')
    ax_ch.plot(t_e, b_e, color='#ff6b6b', linewidth=3, label='Eager')
    ax_ch.set_xlim(0, max_time * 1.05)
    ax_ch.set_ylim(0, NUM_BATCHES * 1.1)
    ax_ch.set_xlabel('Time (s)', fontsize=12, color='#aaa')
    ax_ch.set_ylabel('Batches', fontsize=12, color='#aaa')
    ax_ch.legend(fontsize=12, loc='upper left')
    ax_ch.grid(alpha=0.1)
    ax_ch.tick_params(colors='#666')

    display(fig)
    time.sleep(0.07)

# \u2500\u2500 FINAL REVEAL \u2500\u2500
speedup = eager_cum[-1] / compiled_cum[-1]
eager_ms = np.sum(eager_times) * 1000
compiled_ms = np.sum(compiled_times) * 1000

clear_output(wait=True)
display(HTML(f'''
<div style="text-align:center; padding:50px 40px; background:linear-gradient(135deg, #0d1117 0%, #161b22 30%, #1a472a 70%, #0d1117 100%); border-radius:20px; margin:10px 0; border: 1px solid #30363d;">
  <p style="color:#51cf66; font-size:14px; letter-spacing:4px; margin:0 0 10px 0; text-transform:uppercase;">Triton Kernel Fusion Result</p>
  <h1 style="color:#ffd700; margin:0; font-size:72px; text-shadow: 0 0 30px #ffd70044;">\u26a1 {speedup:.1f}x FASTER</h1>
  <p style="color:#888; font-size:16px; margin:15px 0 25px 0;">torch.compile + Triton GPU Kernels \u2014 {gpu_name}</p>
  <hr style="border-color:#30363d; margin:0 80px 25px 80px;">
  <div style="display:flex; justify-content:center; gap:80px;">
    <div>
      <p style="color:#ff6b6b; font-size:36px; margin:0; font-weight:bold;">{eager_ms:.1f} ms</p>
      <p style="color:#666; font-size:13px; margin:5px 0 0 0;">\U0001f40c Eager Mode</p>
    </div>
    <div style="color:#30363d; font-size:48px; align-self:center;">\u2192</div>
    <div>
      <p style="color:#51cf66; font-size:36px; margin:0; font-weight:bold;">{compiled_ms:.1f} ms</p>
      <p style="color:#666; font-size:13px; margin:5px 0 0 0;">\U0001f680 Compiled + Fused</p>
    </div>
  </div>
  <p style="color:#333; font-size:11px; margin-top:30px;">{NUM_BATCHES} forward passes \u2502 Total time \u2502 Device: {device.upper()} \u2502 GPU: {gpu_name} \u2502 Model: GPT-2 ({num_params:.0f}M params)</p>
</div>
'''))